# 02 - SIP 기초와 SIP 메시지 Mock

이 노트북에서는:
- SIP 메시지의 기본 구조를 배웁니다
- Mock SIP 메시지를 생성하는 방법을 배웁니다
- SIP 의사변수(pseudo-variables)에 대해 알아봅니다

---

## SIP 메시지란?

SIP 메시지는 HTTP와 비슷한 텍스트 기반 프로토콜입니다:

```
INVITE sip:bob@example.com SIP/2.0
Via: SIP/2.0/UDP pc33.example.com;branch=z9hG4bK776
From: "Alice" <sip:alice@example.com>;tag=1928301774
To: <sip:bob@example.com>
Call-ID: a84b4c76e66710@pc33.example.com
CSeq: 314159 INVITE
Contact: <sip:alice@pc33.example.com>
Content-Length: 0
```

주요 메서드:
- **INVITE** — 전화 걸기 (통화 시작)
- **REGISTER** — 전화기가 서버에 등록
- **BYE** — 전화 끊기
- **ACK** — INVITE에 대한 확인 응답

## 1. Mock INVITE 생성

`%%sip INVITE` 매직으로 가짜 INVITE 메시지를 만들 수 있습니다.
이 메시지는 실제 SIP 메시지와 동일한 구조를 가집니다.

In [1]:
%%sip INVITE
From: "Alice" <sip:1001@example.com>;tag=abc123
To: <sip:1002@example.com>
Contact: <sip:1001@192.168.1.100:5060>

Mock INVITE message created:

INVITE sip:1002@example.com SIP/2.0
From: "Alice" <sip:1001@example.com>;tag=abc123
To: <sip:1002@example.com>
Call-ID: a4eefa69@mock
CSeq: 1 INVITE
Contact: <sip:1001@192.168.1.100:5060>

Variables initialized:
  $Ri = "127.0.0.1"
  $Rp = 5060
  $ci = "a4eefa69@mock"
  $cl = "0"
  $cs = "1"
  $ct = "<sip:1001@192.168.1.100:5060>"
  $ft = "abc123"
  $fu = "sip:1001@example.com"
  $rm = "INVITE"
  $ru = "sip:1002@example.com"
  $si = "127.0.0.1"
  $sp = 5060
  $tu = "sip:1002@example.com"
  $ua = ""


## 2. SIP 의사변수 (Pseudo-variables)

SIP 메시지가 생성되면 다음 변수들이 자동으로 설정됩니다:

| 변수 | 의미 | 예시 값 |
|------|------|----------|
| `$ru` | Request-URI | `sip:1002@example.com` |
| `$fu` | From URI | `sip:1001@example.com` |
| `$tu` | To URI | `sip:1002@example.com` |
| `$rm` | Request Method | `INVITE` |
| `$ci` | Call-ID | (자동 생성) |
| `$ft` | From tag | `abc123` |

In [2]:
%%vars

Variable             Type       Value
------------------------------------------------------------
$Ri                  string     "127.0.0.1"
$Rp                  integer    5060
$ci                  string     "a4eefa69@mock"
$cl                  string     "0"
$cs                  string     "1"
$ct                  string     "<sip:1001@192.168.1.100:5060>"
$ft                             "abc123"
$fu                  string     "sip:1001@example.com"
$rm                  string     "INVITE"
$ru                  string     "sip:1002@example.com"
$si                  string     "127.0.0.1"
$sp                  integer    5060
$tu                  string     "sip:1002@example.com"
$ua                  string     ""


## 3. 변수 읽기 및 조작

메시지 변수를 직접 수정할 수 있습니다. 실제 Kamailio에서 이 작업은
메시지를 다른 목적지로 라우팅할 때 사용합니다.

In [ ]:
# Request-URI 변경 (다른 목적지로 라우팅)
$ru = "sip:1002@10.60.91.199:5060";
xlog("New destination: $ru");

## 4. Mock REGISTER 생성

REGISTER는 전화기가 서버에 "나 여기 있어!"라고 알리는 메시지입니다.

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg001
To: <sip:1001@example.com>
Contact: <sip:1001@192.168.1.100:5060;expires=3600>

In [ ]:
# REGISTER 메시지의 메서드 확인
xlog("Method: $rm, From: $fu, Contact: $ct");

## 5. 변수 타입 이해하기

Kamailio cfg의 변수는 두 가지 타입만 있습니다:
- **string**: 문자열
- **integer**: 정수

타입은 값에 따라 자동 결정됩니다.

In [ ]:
$var(str_port) = "5060";   # 문자열 "5060"
$var(int_port) = 5060;     # 정수 5060

xlog("String: $var(str_port), Integer: $var(int_port)");

## 6. $var vs $avp

| 구분 | `$var` | `$avp` |
|------|--------|--------|
| 스코프 | 프로세스 로컬 | 트랜잭션 공유 |
| 값 저장 | 1개만 | 여러 개 (스택) |
| 용도 | 임시 계산, 플래그 | 트랜잭션 간 데이터 전달 |

In [ ]:
$var(tmp) = "local only";
$avp(caller_id) = "1001";
$avp(caller_id) = "1002";  # 스택에 push

%%vars

---

### 요약

- `%%sip` 매직으로 SIP 메시지를 mock 생성
- `$ru`, `$fu`, `$rm` 등의 의사변수로 메시지 정보에 접근
- `$var`는 로컬 임시 변수, `$avp`는 트랜잭션 공유 변수

다음 노트북: **03-routing-basics.ipynb** →